# QNN Data Re-uploading Regression

Notebook này xây dựng một **Variational Quantum Circuit** mô phỏng bằng NumPy để học bài toán regression phi tuyến có cấu trúc Fourier:

$$
y(x)=\sin(2x)+0.5\sin(6x)+0.25\cos(10x), \quad x \in [-\pi, \pi]
$$

Mục tiêu khoa học là kiểm tra QNN với **angle encoding**, **data re-uploading**, **trainable rotations**, **entanglement** và **expectation-value measurement** trên cùng dataset/split/metrics với baseline classical. Notebook không giả định QNN chắc chắn tốt hơn classical model.


## 1. Imports

Notebook dùng NumPy để mô phỏng state vector và dùng Matplotlib để trực quan hóa kết quả.


In [ ]:
import time
import copy
import numpy as np
import matplotlib.pyplot as plt


## 2. Configuration

Các hằng số dataset giống notebook classical để đảm bảo cùng dữ liệu và cùng split.


In [ ]:
SEED = 42
N_SAMPLES = 400
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
DOMAIN_MIN = -np.pi
DOMAIN_MAX = np.pi

rng = np.random.default_rng(SEED)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

N_QUBITS = 3
N_REUPLOAD_LAYERS = 4
ENCODING_FREQUENCIES = np.array([1.0, 2.0, 3.0])


## 3. Reproducibility và Dataset Generation

Dataset và split được sinh bằng đúng seed như notebook classical:

- `SEED = 42` cho dữ liệu.
- `SEED + 1` cho train/validation/test indices.

Checksum của split được in ra để kiểm tra nhanh hai notebook đang dùng cùng phân hoạch dữ liệu.


In [ ]:
def target_function(x):
    return np.sin(2 * x) + 0.5 * np.sin(6 * x) + 0.25 * np.cos(10 * x)


def make_fourier_dataset(n_samples=N_SAMPLES, seed=SEED):
    data_rng = np.random.default_rng(seed)
    x = data_rng.uniform(DOMAIN_MIN, DOMAIN_MAX, size=n_samples)
    y = target_function(x)
    return x.reshape(-1, 1), y.reshape(-1, 1)


def make_split_indices(n_samples=N_SAMPLES, seed=SEED):
    split_rng = np.random.default_rng(seed + 1)
    indices = split_rng.permutation(n_samples)

    n_train = int(TRAIN_RATIO * n_samples)
    n_validation = int(VALIDATION_RATIO * n_samples)

    train_indices = indices[:n_train]
    validation_indices = indices[n_train:n_train + n_validation]
    test_indices = indices[n_train + n_validation:]

    return train_indices, validation_indices, test_indices


x_all, y_all = make_fourier_dataset()
train_indices, validation_indices, test_indices = make_split_indices()

x_train, y_train = x_all[train_indices], y_all[train_indices]
x_validation, y_validation = x_all[validation_indices], y_all[validation_indices]
x_test, y_test = x_all[test_indices], y_all[test_indices]

x_plot = np.linspace(DOMAIN_MIN, DOMAIN_MAX, 800).reshape(-1, 1)
y_plot = target_function(x_plot)

print('Dataset shape:', x_all.shape, y_all.shape)
print('Train / validation / test:', len(x_train), len(x_validation), len(x_test))
print('Split checksum:', int(train_indices.sum()), int(validation_indices.sum()), int(test_indices.sum()))


## 4. Visualization of Dataset

Hình dưới cho thấy target function thật và ba tập train/validation/test.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x_plot, y_plot, color='black', linewidth=2, label='Ground truth')
plt.scatter(x_train, y_train, s=16, alpha=0.65, label='train')
plt.scatter(x_validation, y_validation, s=20, alpha=0.75, label='validation')
plt.scatter(x_test, y_test, s=20, alpha=0.75, label='test')
plt.title('Fourier-structured regression dataset')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()


## 5. Quantum Model Intuition

Angle encoding đưa dữ liệu cổ điển vào circuit bằng các rotation gate. Vì rotation tạo ra phụ thuộc dạng $\sin$ và $\cos$, expectation value của circuit có thể biểu diễn các thành phần giống Fourier.

Với **data re-uploading**, cùng input $x$ được đưa lại vào nhiều layer. Khi kết hợp với:

- nhiều rotation,
- nhiều trainable parameters,
- entanglement giữa các qubit,
- đo expectation value,

circuit có thể sinh ra một hàm Fourier-like phức tạp hơn. Điều này là inductive bias phù hợp với target function nhiều tần số, nhưng không đồng nghĩa quantum computer tự động nhanh hơn hoặc luôn tốt hơn mô hình classical.


## 6. Gate Definition và State-vector Simulation

Ta mô phỏng state vector bằng NumPy để tránh thêm framework quantum mới vào project. Circuit dùng 3 qubit, 4 re-uploading layers. Mỗi layer:

1. Encode input $x$ bằng $R_y(f_q x)$ và $R_z(0.5 f_q x)$ trên từng qubit.
2. Apply trainable `Rot(theta_1, theta_2, theta_3)` tương đương $R_z(\theta_1)R_y(\theta_2)R_z(\theta_3)$.
3. Apply CNOT ring để tạo entanglement.


In [ ]:
def build_cnot_permutation(control_qubit, target_qubit, n_qubits):
    permutation = np.arange(2 ** n_qubits)
    for basis_index in range(2 ** n_qubits):
        control_bit = (basis_index >> (n_qubits - 1 - control_qubit)) & 1
        if control_bit == 1:
            permutation[basis_index] = basis_index ^ (1 << (n_qubits - 1 - target_qubit))
    return permutation


CNOT_PERMUTATIONS = {
    (0, 1): build_cnot_permutation(0, 1, N_QUBITS),
    (1, 2): build_cnot_permutation(1, 2, N_QUBITS),
    (2, 0): build_cnot_permutation(2, 0, N_QUBITS),
}

Z_EIGENVALUES = []
for qubit_index in range(N_QUBITS):
    eigenvalues = np.ones(2 ** N_QUBITS)
    for basis_index in range(2 ** N_QUBITS):
        bit = (basis_index >> (N_QUBITS - 1 - qubit_index)) & 1
        eigenvalues[basis_index] = 1 if bit == 0 else -1
    Z_EIGENVALUES.append(eigenvalues)


def rotation_y(angle):
    c = np.cos(angle / 2)
    s = np.sin(angle / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)


def rotation_z(angle):
    return np.array(
        [[np.exp(-0.5j * angle), 0], [0, np.exp(0.5j * angle)]],
        dtype=complex,
    )


def zero_state(n_qubits):
    state = np.zeros(2 ** n_qubits, dtype=complex)
    state[0] = 1.0
    return state


def apply_single_qubit_gate(state, gate, target_qubit, n_qubits):
    tensor_state = state.reshape([2] * n_qubits)
    tensor_state = np.moveaxis(tensor_state, target_qubit, 0)
    updated = np.tensordot(gate, tensor_state, axes=([1], [0]))
    updated = np.moveaxis(updated, 0, target_qubit)
    return updated.reshape(-1)


def apply_cnot(state, control_qubit, target_qubit, n_qubits):
    updated = np.zeros_like(state)
    updated[CNOT_PERMUTATIONS[(control_qubit, target_qubit)]] = state
    return updated


def measure_z_expectation(state, qubit_index, n_qubits):
    probabilities = np.abs(state) ** 2
    return float(np.real(np.sum(Z_EIGENVALUES[qubit_index] * probabilities)))


## 7. Model Definition

QNN dùng:

- 3 qubit.
- 4 data re-uploading layers.
- 3 trainable rotation parameters cho mỗi qubit ở mỗi layer.
- Linear readout từ 3 expectation values sang scalar regression output.

Trainable parameters:

- Circuit rotations: $4 \times 3 \times 3 = 36$
- Readout weights: $3$
- Readout bias: $1$

Tổng cộng: **40 trainable parameters**.


In [ ]:
def initialize_qnn_parameters(random_generator):
    return {
        'theta': random_generator.normal(0, 0.15, size=(N_REUPLOAD_LAYERS, N_QUBITS, 3)),
        'readout_w': random_generator.normal(0, 0.2, size=(N_QUBITS, 1)),
        'readout_b': np.zeros((1, 1)),
    }


def count_parameters(params):
    return int(sum(value.size for value in params.values()))


def apply_data_encoding(state, x_scalar):
    for qubit_index, frequency in enumerate(ENCODING_FREQUENCIES):
        state = apply_single_qubit_gate(
            state,
            rotation_y(frequency * x_scalar),
            qubit_index,
            N_QUBITS,
        )
        state = apply_single_qubit_gate(
            state,
            rotation_z(0.5 * frequency * x_scalar),
            qubit_index,
            N_QUBITS,
        )
    return state


def apply_trainable_layer(state, layer_params):
    for qubit_index in range(N_QUBITS):
        phi, theta, omega = layer_params[qubit_index]
        state = apply_single_qubit_gate(state, rotation_z(phi), qubit_index, N_QUBITS)
        state = apply_single_qubit_gate(state, rotation_y(theta), qubit_index, N_QUBITS)
        state = apply_single_qubit_gate(state, rotation_z(omega), qubit_index, N_QUBITS)
    return state


def apply_entanglement_ring(state):
    state = apply_cnot(state, control_qubit=0, target_qubit=1, n_qubits=N_QUBITS)
    state = apply_cnot(state, control_qubit=1, target_qubit=2, n_qubits=N_QUBITS)
    state = apply_cnot(state, control_qubit=2, target_qubit=0, n_qubits=N_QUBITS)
    return state


def circuit_expectations(x_scalar, theta):
    state = zero_state(N_QUBITS)
    for layer_index in range(N_REUPLOAD_LAYERS):
        state = apply_data_encoding(state, x_scalar)
        state = apply_trainable_layer(state, theta[layer_index])
        state = apply_entanglement_ring(state)

    return np.array([
        measure_z_expectation(state, qubit_index, N_QUBITS)
        for qubit_index in range(N_QUBITS)
    ])


def qnn_expectation_features(x, theta):
    return np.array([circuit_expectations(float(x_value), theta) for x_value in x[:, 0]])


def predict_qnn(x, params):
    expectation_features = qnn_expectation_features(x, params['theta'])
    return expectation_features @ params['readout_w'] + params['readout_b']


qnn_params = initialize_qnn_parameters(rng)
print('QNN trainable parameters:', count_parameters(qnn_params))


## 8. Parameter Shift Gradient

QNN sử dụng parameter-shift rule cho các trainable quantum rotation gates:

$$
\frac{\partial f}{\partial \theta}
=
\frac{
f(\theta+\pi/2)-f(\theta-\pi/2)
}{2}
$$

Trong code dưới, $f$ là prediction của circuit. Gradient của MSE được tính bằng chain rule:

$$
\frac{\partial L}{\partial \theta}
=
\frac{1}{N}\sum_i 2(\hat{y}_i-y_i)
\frac{\partial \hat{y}_i}{\partial \theta}
$$

Parameter shift là phương pháp gradient phù hợp cho parameterized quantum gates, nhưng không được hiểu là luôn nhanh hơn backpropagation classical.


In [ ]:
def qnn_gradients(x_batch, y_batch, params):
    predictions = predict_qnn(x_batch, params)
    residual_gradient = 2 * (predictions - y_batch) / len(x_batch)

    expectation_features = qnn_expectation_features(x_batch, params['theta'])
    grads = {
        'theta': np.zeros_like(params['theta']),
        'readout_w': expectation_features.T @ residual_gradient,
        'readout_b': np.sum(residual_gradient, axis=0, keepdims=True),
    }

    shift = np.pi / 2
    iterator = np.ndindex(params['theta'].shape)

    for param_index in iterator:
        theta_plus = params['theta'].copy()
        theta_minus = params['theta'].copy()
        theta_plus[param_index] += shift
        theta_minus[param_index] -= shift

        params_plus = {
            'theta': theta_plus,
            'readout_w': params['readout_w'],
            'readout_b': params['readout_b'],
        }
        params_minus = {
            'theta': theta_minus,
            'readout_w': params['readout_w'],
            'readout_b': params['readout_b'],
        }

        pred_plus = predict_qnn(x_batch, params_plus)
        pred_minus = predict_qnn(x_batch, params_minus)
        d_prediction = 0.5 * (pred_plus - pred_minus)
        grads['theta'][param_index] = np.sum(residual_gradient * d_prediction)

    return grads


## 9. Training

QNN dùng MSE loss và Adam optimizer. Do parameter-shift cần nhiều lần chạy circuit cho mỗi trainable quantum gate, mỗi epoch chỉ lấy một stochastic mini-batch từ train split để cập nhật tham số. Train/validation MSE vẫn được log trên toàn bộ split, best model vẫn được chọn bằng validation MSE, và test set chỉ dùng một lần ở cuối.


In [ ]:
def adam_update(params, grads, optimizer_state, learning_rate, beta1=0.9, beta2=0.999, epsilon=1e-8):
    optimizer_state['t'] += 1
    t = optimizer_state['t']

    for key in params:
        optimizer_state['m'][key] = beta1 * optimizer_state['m'][key] + (1 - beta1) * grads[key]
        optimizer_state['v'][key] = beta2 * optimizer_state['v'][key] + (1 - beta2) * (grads[key] ** 2)

        m_hat = optimizer_state['m'][key] / (1 - beta1 ** t)
        v_hat = optimizer_state['v'][key] / (1 - beta2 ** t)
        params[key] -= learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)

    return params


def make_adam_state(params):
    return {
        't': 0,
        'm': {key: np.zeros_like(value) for key, value in params.items()},
        'v': {key: np.zeros_like(value) for key, value in params.items()},
    }


def sample_minibatch(x, y, random_generator, batch_size):
    batch_indices = random_generator.choice(len(x), size=batch_size, replace=False)
    return x[batch_indices], y[batch_indices]


def mean_squared_error(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


def train_qnn(params, epochs=20, batch_size=8, learning_rate=0.03):
    optimizer_state = make_adam_state(params)
    history = {'train_mse': [], 'validation_mse': []}
    best_params = copy.deepcopy(params)
    best_validation_mse = float('inf')
    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        x_batch, y_batch = sample_minibatch(x_train, y_train, rng, batch_size)
        grads = qnn_gradients(x_batch, y_batch, params)
        params = adam_update(params, grads, optimizer_state, learning_rate)

        train_pred = predict_qnn(x_train, params)
        validation_pred = predict_qnn(x_validation, params)
        train_mse = mean_squared_error(y_train, train_pred)
        validation_mse = mean_squared_error(y_validation, validation_pred)

        history['train_mse'].append(train_mse)
        history['validation_mse'].append(validation_mse)

        if validation_mse < best_validation_mse:
            best_validation_mse = validation_mse
            best_params = copy.deepcopy(params)

        if epoch == 1 or epoch % 10 == 0:
            print(f'Epoch {epoch:03d} | train MSE = {train_mse:.6f} | validation MSE = {validation_mse:.6f}')

    training_time = time.perf_counter() - start_time
    return best_params, history, best_validation_mse, training_time


best_qnn_params, history, best_validation_mse, training_time = train_qnn(qnn_params)


## 10. Evaluation Metrics

Cuối cùng đánh giá trên test set bằng MSE, MAE và $R^2$.


In [ ]:
def mean_squared_error(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


def mean_absolute_error(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def r2_score_numpy(y_true, y_pred):
    ss_residual = np.sum((y_true - y_pred) ** 2)
    ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1 - ss_residual / ss_total)


In [ ]:
test_predictions = predict_qnn(x_test, best_qnn_params)
test_mse = mean_squared_error(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)
test_r2 = r2_score_numpy(y_test, test_predictions)

print('------------------------------------')
print('Model: QNN Data Re-uploading')
print(f'Trainable parameters: {count_parameters(best_qnn_params)}')
print(f'Best validation MSE: {best_validation_mse:.6f}')
print(f'Test MSE: {test_mse:.6f}')
print(f'Test MAE: {test_mae:.6f}')
print(f'Test R^2: {test_r2:.6f}')
print(f'Training time: {training_time:.2f} seconds')
print('------------------------------------')


## 11. Visualization

Ba hình bắt buộc:

1. Ground Truth vs Prediction.
2. Training Curve.
3. Prediction Error.


In [ ]:
plot_predictions = predict_qnn(x_plot, best_qnn_params)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(x_plot, y_plot, color='black', linewidth=2, label='Ground truth')
axes[0].plot(x_plot, plot_predictions, color='tab:orange', linewidth=2, label='Prediction')
axes[0].scatter(x_train, y_train, s=14, alpha=0.45, label='train')
axes[0].scatter(x_test, y_test, s=18, alpha=0.75, label='test')
axes[0].set_title('QNN Data Re-uploading Regression')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()

axes[1].plot(history['train_mse'], label='Train MSE', linewidth=2)
axes[1].plot(history['validation_mse'], label='Validation MSE', linewidth=2)
axes[1].set_title('Training Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].legend()

sorted_test = np.argsort(x_test[:, 0])
sorted_x_test = x_test[sorted_test]
sorted_abs_error = np.abs(y_test[sorted_test] - test_predictions[sorted_test])
axes[2].plot(sorted_x_test, sorted_abs_error, marker='o', linewidth=1.5)
axes[2].set_title('Prediction Error')
axes[2].set_xlabel('x')
axes[2].set_ylabel('|y_true - y_pred|')

plt.tight_layout()
plt.show()


## 12. Results Summary và Scientific Interpretation

QNN trong notebook này dùng data re-uploading để đưa cùng input $x$ vào nhiều layer. Điều này giúp expectation value có thêm cơ hội tạo ra các thành phần Fourier-like từ rotation gate. Entanglement ring trộn thông tin giữa các qubit, giúp các feature lượng tử không chỉ là ba kênh độc lập.

Khi phân tích kết quả sau khi chạy notebook, hãy kiểm tra:

- Prediction có bám được các dao động nhanh do $\sin(6x)$ và $\cos(10x)$ không.
- Validation MSE có cải thiện khi train tiếp hay bị dao động/overfit.
- Prediction error tập trung ở vùng biên hay vùng có dao động nhanh.

Không kết luận quantum advantage chỉ từ kiến trúc. Nếu QNN đạt test MSE thấp hơn baseline classical với số parameters tương đương, kết luận hợp lệ là: trong thí nghiệm này, QNN data re-uploading có inductive bias phù hợp với regression Fourier hơn. Nếu không, cần ghi trung thực rằng chưa quan sát được advantage trong cấu hình hiện tại.
